#Проект по Программированию на Питоне для Бизнес-аналитики.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise SystemExit(
        "Файл .env не найден.\n"
        "Создайте его в корне репозитория (рядом с README.md) и положите туда\n"
        "строку PRIME_DSN=... из бота курса."
    )

load_dotenv(env_path)
dsn = os.environ.get("PRIME_DSN")
if not dsn:
    raise SystemExit(f"Файл {env_path} нашёлся, но переменной PRIME_DSN в нём нет.")
print("логин:", dsn.split("//", 1)[1].split(":", 1)[0])

логин: edanikeev_1


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

engine = create_engine(dsn)

pd.read_sql("select current_user, current_database(), version()", engine).T

,0
current_user,edanikeev_1
current_database,prime
version,PostgreSQL 18.6 (Ubuntu 18.6-0ubuntu0.26.04.1)...


In [3]:
pd.read_sql("""
select c.relname                                     as "таблица",
       to_char(c.reltuples::bigint, 'FM999 999 999') as "строк, примерно",
       pg_size_pretty(pg_total_relation_size(c.oid)) as "на диске"
from pg_class c
join pg_namespace n on n.oid = c.relnamespace
where n.nspname = 'prime' and c.relkind = 'r'
order by c.reltuples desc
""", engine)

,таблица,"строк, примерно",на диске
0,service_usage,7 969 173,1232 MB
1,transactions,5 974 746,954 MB
2,payments,3 147 501,471 MB
3,cashback,2 204 058,358 MB
4,subscription_members,555 858,61 MB
5,clients,500 000,74 MB
6,subscriptions,333 905,58 MB
7,support_tickets,150 000,38 MB
8,marketing_spend,3 606,536 kB
9,assignments,58,96 kB


In [6]:
pd.set_option('display.max_colwidth', None)

pd.read_sql("""
select table_name,
       string_agg(column_name || ' (' || data_type || ')', ', ' order by ordinal_position) as columns
from information_schema.columns
where table_schema = 'prime'
group by table_name
order by table_name
""", engine)

,table_name,columns
0,assignments,"login (text), table_name (text), segment_column (text), segment (text), period_start (date), period_end (date), client_filter (text)"
1,cashback,"cashback_id (uuid), transaction_id (uuid), accrued_at (timestamp with time zone), bonus_amount (numeric), rate_applied (numeric)"
2,clients,"client_id (uuid), registered_at (timestamp with time zone), region_code (integer), time_zone (text), birth_year (integer), acquisition_channel (text), kyc_level (integer)"
3,marketing_spend,"spend_id (integer), channel (text), spend_date (date), amount (numeric), impressions (bigint), clicks (bigint)"
4,partners,"partner_id (integer), partner_name (text), partner_type (text)"
5,payments,"payment_id (uuid), subscription_id (uuid), paid_at (timestamp with time zone), amount (numeric), status (text), payment_method (text)"
6,service_usage,"usage_id (uuid), client_id (uuid), service_id (integer), started_at (timestamp with time zone), ended_at (timestamp with time zone), device_type (text)"
7,services,"service_id (integer), service_name (text), category (text), available_from_tariff (integer)"
8,subscription_members,"subscription_id (uuid), client_id (uuid), joined_at (timestamp with time zone), left_at (timestamp with time zone)"
9,subscriptions,"subscription_id (uuid), owner_client_id (uuid), tariff_id (integer), started_at (timestamp with time zone), ended_at (timestamp with time zone), is_trial (boolean), partner_id (integer), cancel_reason (text)"


In [5]:
pd.read_sql("""
select count(*)                     as rows_after_join,
       count(distinct p.payment_id) as unique_payments
from prime.payments p
join prime.subscription_members m on m.subscription_id = p.subscription_id
""", engine)

,rows_after_join,unique_payments
0,5248649,2204123


In [8]:
pd.read_sql("select * from prime.assignments where login = current_user", engine)

,login,table_name,segment_column,segment,period_start,period_end,client_filter
0,edanikeev_1,support_tickets,category,доступ к сервису,2025-04-01,2025-09-30,client_id in (\n select client_id\n from prime.support_tickets\n where category = 'доступ к сервису'\n and created_at >= '2025-04-01 00:00+00'\n and created_at < '2025-10-01 00:00+00'\n)


In [10]:
pd.read_sql("select * from prime.tariffs order by tariff_id, valid_from", engine)

,tariff_id,tariff_name,monthly_fee,cashback_rate,share_slots,valid_from,valid_to
0,1,Базовый,199.0,0.01,3,2024-01-01,2026-02-28
1,1,Базовый,249.0,0.01,3,2026-03-01,2030-12-31
2,2,Расширенный,399.0,0.03,5,2024-01-01,2026-02-28
3,2,Расширенный,429.0,0.03,5,2026-02-01,2026-06-30
4,2,Расширенный,449.0,0.03,5,2026-03-01,2030-12-31


In [11]:
pd.read_sql("""
select status, count(*) as n, sum(amount) as amount
from prime.payments
group by status
""", engine)

,status,n,amount
0,failed,104140,30520660.0
1,refunded,37934,11133066.0
2,success,3005335,880737565.0


In [12]:
SEG = """
seg as (
    select distinct client_id
    from prime.support_tickets
    where category = 'доступ к сервису'
      and created_at >= '2025-04-01 00:00+00'
      and created_at <  '2025-10-01 00:00+00'
)
"""

In [13]:
tickets = pd.read_sql("""
select date_trunc('month', created_at at time zone 'UTC')::date as month,
       category,
       count(*) as tickets
from prime.support_tickets
group by 1, 2
order by 1, 2
""", engine)
tickets.pivot(index='month', columns='category', values='tickets')

category,доступ к сервису,кэшбэк,оплата,отмена подписки,прочее,шеринг
month,,,,,,
2025-01-01,697,614,1017,514,315,511
2025-02-01,855,702,1119,546,424,615
2025-03-01,1011,819,1244,680,483,713
2025-04-01,1081,873,1384,667,449,793
2025-05-01,1191,884,1553,707,544,788
2025-06-01,1237,953,1578,737,526,792
2025-07-01,1462,1053,1753,781,553,903
2025-08-01,1486,1123,1874,893,608,995
2025-09-01,1451,1086,1932,915,604,976


In [14]:
pd.read_sql("""
select created_at, text
from prime.support_tickets
where category = 'доступ к сервису'
  and created_at >= '2025-04-01 00:00+00' and created_at < '2025-10-01 00:00+00'
order by created_at
limit 30
""", engine)

,created_at,text
0,2025-04-01 02:53:50+00:00,Приложение не видит подписку после переустановки.
1,2025-04-01 03:03:09+00:00,"Купил расширенный тариф, но каршеринг всё равно недоступен."
2,2025-04-01 03:21:14+00:00,"Купил расширенный тариф, но каршеринг всё равно недоступен."
3,2025-04-01 03:33:22+00:00,Приложение не видит подписку после переустановки.
4,2025-04-01 04:42:15+00:00,"Купил расширенный тариф, но каршеринг всё равно недоступен."
5,2025-04-01 04:59:41+00:00,"Не открывается сервис, пишет что подписка не активна."
6,2025-04-01 05:40:34+00:00,"Купил расширенный тариф, но каршеринг всё равно недоступен."
7,2025-04-01 05:49:58+00:00,"Не открывается сервис, пишет что подписка не активна."
8,2025-04-01 07:23:13+00:00,"Не открывается сервис, пишет что подписка не активна."
9,2025-04-01 09:07:43+00:00,Приложение не видит подписку после переустановки.


In [15]:
pd.read_sql(f"""
with {SEG}
select count(*) as clients,
       count(*) filter (where client_id in (select owner_client_id from prime.subscriptions))  as owners,
       count(*) filter (where client_id in (select client_id from prime.subscription_members)) as in_members
from seg
""", engine)

,clients,owners,in_members
0,7775,5446,5247


In [16]:
pd.read_sql(f"""
with {SEG}
select date_trunc('month', p.paid_at at time zone 'UTC')::date as month,
       p.status,
       count(distinct s.owner_client_id) as payers,
       count(*)                          as payments,
       sum(p.amount)                     as revenue
from prime.payments p
join prime.subscriptions s on s.subscription_id = p.subscription_id
where s.owner_client_id in (select client_id from seg)
group by 1, 2
order by 1, 2
""", engine)

,month,status,payers,payments,revenue
0,2025-01-01,failed,82,82,23918.0
1,2025-01-01,refunded,33,33,9967.0
2,2025-01-01,success,2327,2327,637673.0
3,2025-02-01,failed,98,98,27102.0
4,2025-02-01,refunded,41,41,11159.0
5,2025-02-01,success,2712,2717,743883.0
6,2025-03-01,failed,108,110,30490.0
7,2025-03-01,refunded,42,42,12558.0
8,2025-03-01,success,3151,3256,889544.0
9,2025-04-01,failed,124,124,34276.0


In [17]:
retention = pd.read_sql(f"""
with {SEG},
base as (   -- владельцы с успешным платежом в период варианта
    select distinct s.owner_client_id as client_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
      and p.paid_at >= '2025-04-01 00:00+00'
      and p.paid_at <  '2025-10-01 00:00+00'
),
groups as ( -- сегмент и контроль
    select client_id,
           case when client_id in (select client_id from seg)
                then 'обращались про доступ' else 'контроль' end as grp
    from base
),
sizes as (
    select grp, count(*) as base_size from groups group by grp
),
paid as (   -- кто из владельцев платил в каждом месяце
    select distinct date_trunc('month', p.paid_at at time zone 'UTC')::date as month,
           s.owner_client_id as client_id
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
)
select pm.month, g.grp, count(*) as payers, max(sz.base_size) as base_size
from groups g
join sizes sz on sz.grp = g.grp
join paid pm on pm.client_id = g.client_id
group by 1, 2
order by 1, 2
""", engine)

retention['share'] = retention['payers'] / retention['base_size']
retention.pivot(index='month', columns='grp', values='share').round(3)

grp,контроль,обращались про доступ
month,,
2025-01-01,0.419,0.468
2025-02-01,0.508,0.565
2025-03-01,0.603,0.681
2025-04-01,0.658,0.731
2025-05-01,0.697,0.775
2025-06-01,0.723,0.795
2025-07-01,0.761,0.818
2025-08-01,0.793,0.819
2025-09-01,0.815,0.801


In [18]:
pd.read_sql("select * from prime.services order by service_id", engine)

,service_id,service_name,category,available_from_tariff
0,1,Кино,видео,1
1,2,Сериалы+,видео,2
2,3,Музыка,музыка,1
3,4,Аудиокниги,музыка,2
4,5,Продукты,доставка,1
5,6,Экспресс-доставка,доставка,2
6,7,Каршеринг,каршеринг,2
7,8,Самокаты,каршеринг,1
8,9,Аптека,аптека,1
9,10,Телемедицина,аптека,2


In [19]:
pd.read_sql("select min(paid_at), max(paid_at) from prime.payments", engine)

,min,max
0,2025-01-06 00:43:16+00:00,2026-08-30 23:27:52+00:00


In [20]:
pd.read_sql("""
select date_trunc('month', created_at at time zone 'UTC')::date as month,
       text,
       count(*) as tickets
from prime.support_tickets
where category = 'доступ к сервису'
group by 1, 2
order by 1, 2
""", engine).pivot(index='month', columns='text', values='tickets')

text,"Купил расширенный тариф, но каршеринг всё равно недоступен.","Не открывается сервис, пишет что подписка не активна.",Приложение не видит подписку после переустановки.
month,,,
2025-01-01,215,241,241
2025-02-01,274,285,296
2025-03-01,319,353,339
2025-04-01,374,384,323
2025-05-01,379,414,398
2025-06-01,424,410,403
2025-07-01,485,484,493
2025-08-01,501,507,478
2025-09-01,486,470,495


In [21]:
pd.read_sql("""
select st.text,
       coalesce(t.tariff_name, 'нет своей активной подписки') as tariff_at_ticket,
       count(distinct st.ticket_id) as tickets
from prime.support_tickets st
left join prime.subscriptions s
       on s.owner_client_id = st.client_id
      and s.started_at <= st.created_at
      and (s.ended_at is null or s.ended_at > st.created_at)
left join (select distinct tariff_id, tariff_name from prime.tariffs) t
       on t.tariff_id = s.tariff_id
where st.category = 'доступ к сервису'
  and st.created_at >= '2025-04-01 00:00+00'
  and st.created_at <  '2025-10-01 00:00+00'
group by 1, 2
order by 1, 3 desc
""", engine).pivot(index='tariff_at_ticket', columns='text', values='tickets')

text,"Купил расширенный тариф, но каршеринг всё равно недоступен.","Не открывается сервис, пишет что подписка не активна.",Приложение не видит подписку после переустановки.
tariff_at_ticket,,,
Базовый,774,788,789
Расширенный,478,486,483
нет своей активной подписки,1397,1395,1318


## Итог разведки

- **Выручка** — только `status = 'success'`. Клиента к платежу привязываю через `subscriptions.owner_client_id`. Соединение с `subscription_members` размножает платежи в 2,4 раза и одновременно теряет 30% из них (подписки без участников).
- **Справочник цен.** Строка 429 ₽ перекрывает соседние, и ни одного платежа на 429 ₽ в базе нет. Цену беру из `payments.amount`, справочник для расчёта не нужен.
- **Гипотеза по сегменту варианта не подтвердилась.** Клиенты, писавшие про доступ к сервису в апреле–сентябре 2025, после периода платят так же, как остальные: в июле 2026 доля платящих 0,554 против 0,553 в контроле. Тексты жалоб от тарифа не зависят.
- **На всей базе нашлось событие крупнее — 22–23 августа 2026 года.** Проект ниже — про него.

# Проект 1. Почему в конце августа 2026 года упала выручка ПРАЙМ

**Бизнес-вопрос.** В последнюю неделю августа 2026 года выручка ПРАЙМ в день заметно упала. Где именно, за счёт чего, почему — и что с этим делать?

**План** — схема семинара 6: сначала «где», потом «за счёт чего».

1. Что случилось — выручка по дням.
2. Где — аддитивная модель по каналам привлечения.
3. За счёт чего — мультипликативная модель: активные подписки × платежей на подписку × цена платежа.
4. Кого задело — витрина отключённых клиентов и протокол сборки.
5. Почему — и чего мы не знаем.
6. Вывод и что делать.

**Окна сравнения: w0 = 17–21.08 и w1 = 24–28.08, оба с понедельника по пятницу.**

- Соседние недели: база растёт примерно на 2% в месяц, за неделю рост почти не искажает сравнение.
- Одинаковая длина и одинаковые дни недели — суммы сравниваю напрямую, без пересчёта на день.
- Ни одно окно не задевает 22–23.08, дни самого события. На семинаре 6 окно 17–22.08 включало субботу 22-го, когда половина подписок уже была отключена.
- 30.08 в данных неполный (платежей вдвое меньше обычного), 31.08 нет вовсе — их не беру.

In [ ]:
import itertools
import math

import matplotlib.dates as mdates
import matplotlib.pyplot as plt


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Сутки считаем по UTC, как на семинарах."""
    with engine.connect() as conn:
        conn.execute(text("set time zone 'UTC'"))
        return pd.read_sql(text(sql), conn, params=params)


W0 = ("2026-08-17", "2026-08-22")      # пн 17.08 – пт 21.08; правый конец не включён
W1 = ("2026-08-24", "2026-08-29")      # пн 24.08 – пт 28.08
EVENT = ("2026-08-22", "2026-08-24")   # сутки массового отключения: 22 и 23 августа
LAST_DAY = "2026-08-30"                # 30.08 неполный — берём данные строго до него

for w in (W0, W1):
    assert pd.Timestamp(w[0]).day_name() == "Monday", "окно должно начинаться с понедельника"

ACCENT, GREY, INK, GRID = "#e34948", "#a8a7a1", "#52514e", "#e5e4df"   # один акцент, остальное серое


def rub(x: float) -> str:
    """12345678.9 -> '12 345 679'."""
    return f"{x:,.0f}".replace(",", " ")


def signed(x: float) -> str:
    """-1234.5 -> '−1 235', 12 -> '+12': со знаком и пробелами между разрядами."""
    return f"{x:+,.0f}".replace(",", " ").replace("-", "−")


print("окна:", W0, W1)

## 1. Что случилось

Выручка — сумма `amount` по успешным платежам. Канал — у владельца подписки (`clients.acquisition_channel` через `owner_client_id`). Клиентов без канала не выбрасываю, а показываю отдельной строкой «без канала» — договорённость семинара 6.

Зерно таблицы `daily` — **день × канал**. Оба соединения N:1 (платёж → подписка → владелец), поэтому платежи не размножаются. Это проверяет контрольная сумма, посчитанная по `payments` без единого соединения.

In [ ]:
daily = q("""
    select p.paid_at::date                               as day,
           coalesce(c.acquisition_channel, 'без канала') as channel,
           count(*)                                      as n_payments,
           sum(p.amount)                                 as revenue
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id   -- N:1: у платежа одна подписка
    join prime.clients c       on c.client_id = s.owner_client_id         -- N:1: у подписки один владелец
    where p.status = 'success'
      and p.paid_at >= '2026-07-01' and p.paid_at < :last_day
    group by 1, 2
    order by 1, 2
""", last_day=LAST_DAY)
daily["day"] = pd.to_datetime(daily["day"])

# Независимый контроль: та же сумма прямо по payments, без соединений.
control = q("""
    select sum(amount) from prime.payments
    where status = 'success' and paid_at >= '2026-07-01' and paid_at < :last_day
""", last_day=LAST_DAY).iloc[0, 0]
assert abs(daily["revenue"].sum() - control) < 0.01, "соединения потеряли или размножили платежи"
print("контроль сошёлся:", rub(control), "₽")

by_day = daily.pivot_table(index="day", columns="channel", values="revenue", aggfunc="sum").fillna(0)
by_day["итого"] = by_day.sum(axis=1)
(by_day.loc["2026-08-17":] / 1000).round().astype(int)       # тыс. ₽

In [ ]:
tel = by_day["partner_telecom"]
others = by_day["итого"] - tel

in_w0 = (by_day.index >= W0[0]) & (by_day.index < W0[1])
in_w1 = (by_day.index >= W1[0]) & (by_day.index < W1[1])
tel_change = tel[in_w1].mean() / tel[in_w0].mean() - 1
others_change = others[in_w1].mean() / others[in_w0].mean() - 1

fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ax.plot(others.index, others / 1000, color=GREY, linewidth=2, label="остальные каналы")
ax.plot(tel.index, tel / 1000, color=ACCENT, linewidth=2, label="partner_telecom")
ax.axvspan(pd.Timestamp(EVENT[0]), pd.Timestamp(EVENT[1]), color=ACCENT, alpha=0.12, linewidth=0)
ax.text(pd.Timestamp(EVENT[0]), others.max() / 1000 * 1.04, " 22–23.08", color=ACCENT, va="bottom")
ax.set_ylim(0, others.max() / 1000 * 1.15)
ax.set_ylabel("выручка в день, тыс. ₽")
ax.set_xlabel("июль – август 2026, UTC; 30.08 неполный и не показан")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
others_pct = f"{others_change:+.1%}".replace("-", "−")
ax.set_title(f"После 22–23 августа выручка канала partner_telecom упала на {-tel_change:.0%}, "
             f"у остальных каналов изменение {others_pct}", loc="left")
ax.legend(frameon=False, loc="center left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=GRID)
plt.show()

## 2. Где: аддитивная модель по каналам

Выручка — сумма выручек каналов, значит её изменение — сумма изменений каналов. `assert` проверяет, что части сходятся с целым до копейки.

In [ ]:
def window_sum(df: pd.DataFrame, w: tuple[str, str]) -> pd.Series:
    """Выручка по каналам за окно [w[0], w[1])."""
    part = df[(df["day"] >= w[0]) & (df["day"] < w[1])]
    return part.groupby("channel")["revenue"].sum()


where = pd.DataFrame({"w0": window_sum(daily, W0), "w1": window_sum(daily, W1)}).fillna(0)
where["delta"] = where["w1"] - where["w0"]
where = where.sort_values("delta")

total_delta = where["w1"].sum() - where["w0"].sum()
assert abs(where["delta"].sum() - total_delta) < 0.01, "каналы не сходятся с итогом"

where.loc["итого"] = where.sum()
(where / 1000).round(1)       # тыс. ₽

In [ ]:
def waterfall(deltas: pd.Series, title: str, accent: str) -> plt.Axes:
    """Каскад изменений, как на семинаре 6: столбик на фактор и «итого» в конце.

    Акцентным цветом — только тот столбик, о котором вывод; остальные серые.
    """
    before = deltas.cumsum() - deltas      # уровень, на котором закончился предыдущий столбик
    colors = [ACCENT if name == accent else GREY for name in deltas.index]

    fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
    bars = ax.bar(deltas.index, deltas, bottom=before, color=colors, zorder=2)
    total = ax.bar("итого", deltas.sum(), color=INK, zorder=2)
    for group in (bars, total):
        ax.bar_label(group, fmt=signed, padding=3)
    ax.axhline(0, color=INK, linewidth=0.8)
    ax.set_title(title, loc="left")
    ax.set_ylabel("тыс. ₽")
    ax.use_sticky_edges = False            # иначе подписи упрутся в край графика
    ax.margins(y=0.15)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color=GRID, zorder=0)
    return ax


steps = where.drop("итого")["delta"] / 1000
tel_delta = steps["partner_telecom"]
ax = waterfall(steps, accent="partner_telecom",
               title=f"Почти всё падение — канал partner_telecom: {signed(tel_delta)} из {signed(steps.sum())} тыс. ₽")
ax.set_xlabel("Изменение выручки по каналам: w0 = 17–21.08 → w1 = 24–28.08")
ax.tick_params(axis="x", labelrotation=15)
plt.show()

## 3. За счёт чего: мультипликативная модель

> выручка = активные подписки × платежей на подписку × цена платежа

- **Активна в окне** подписка, которая началась до конца окна и не закончилась до его начала — то же определение, что на семинаре 6 для месяца.
- **Порядок подстановок:** сначала количество (активные подписки), потом частота (платежей на подписку), потом цена — договорённость курса. Ниже проверка всеми шестью порядками.
- Считаю для всей базы, отдельно для partner_telecom и для остальных каналов.

In [ ]:
def factors(w: tuple[str, str]) -> pd.DataFrame:
    """Окно -> по каналам: активные подписки, успешные платежи, выручка."""
    return q("""
        with act as (
            select coalesce(c.acquisition_channel, 'без канала') as channel, count(*) as active
            from prime.subscriptions s
            join prime.clients c on c.client_id = s.owner_client_id
            where s.started_at < cast(:b as timestamptz)
              and (s.ended_at is null or s.ended_at >= cast(:a as timestamptz))
            group by 1
        ),
        rev as (
            select coalesce(c.acquisition_channel, 'без канала') as channel,
                   count(*) as n_payments, sum(p.amount) as revenue
            from prime.payments p
            join prime.subscriptions s on s.subscription_id = p.subscription_id
            join prime.clients c on c.client_id = s.owner_client_id
            where p.status = 'success'
              and p.paid_at >= cast(:a as timestamptz) and p.paid_at < cast(:b as timestamptz)
            group by 1
        )
        select a.channel, a.active,
               coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
        from act a
        left join rev r on r.channel = a.channel
    """, a=w[0], b=w[1]).set_index("channel")


f0, f1 = factors(W0), factors(W1)

# Та же выручка, что в аддитивной модели: две модели считают одно и то же.
assert abs(f0["revenue"].sum() - where.loc["итого", "w0"]) < 0.01
assert abs(f1["revenue"].sum() - where.loc["итого", "w1"]) < 0.01

pd.concat({"w0": f0, "w1": f1}, axis=1)

In [ ]:
def contrib(x0: dict, x1: dict, order: list[str]) -> dict:
    """Цепные подстановки (семинар 6): меняем факторы по одному в порядке order."""
    cur, out = dict(x0), {}
    for f in order:
        before = math.prod(cur.values())
        cur[f] = x1[f]                     # фактор переходит на новое значение, остальные стоят
        out[f] = math.prod(cur.values()) - before
    total = math.prod(x1.values()) - math.prod(x0.values())
    assert abs(sum(out.values()) - total) < 0.01, "сумма влияний не равна изменению"
    return out


def to_factors(f: pd.DataFrame) -> dict:
    """Строки каналов -> три множителя выручки."""
    s = f.sum()
    return {"active": s["active"],
            "per_sub": s["n_payments"] / s["active"],
            "price": s["revenue"] / s["n_payments"]}


ORDER = ["active", "per_sub", "price"]     # сначала «сколько», потом «как часто», потом «почём»
LABELS = {"active": "активные подписки", "per_sub": "платежей на подписку", "price": "цена платежа"}

groups = {
    "вся база": (f0, f1),
    "partner_telecom": (f0.loc[["partner_telecom"]], f1.loc[["partner_telecom"]]),
    "остальные каналы": (f0.drop("partner_telecom"), f1.drop("partner_telecom")),
}

values, effects = {}, {}
for name, (a, b) in groups.items():
    x0, x1 = to_factors(a), to_factors(b)
    values[(name, "w0")], values[(name, "w1")] = x0, x1
    effects[name] = contrib(x0, x1, ORDER)

values = pd.DataFrame(values).T.rename(columns=LABELS)
effects = pd.DataFrame(effects).T.rename(columns=LABELS)
effects["изменение выручки"] = effects.sum(axis=1)

assert abs(effects.loc["вся база", "изменение выручки"] - total_delta) < 0.01, "не сходится с аддитивной моделью"

display(values.round(4))
(effects / 1000).round(1)       # тыс. ₽

**Как читать.** Почти всё изменение — фактор «активные подписки», и почти весь он — partner_telecom.

Два вопроса, которые зададут на защите:

- *Почему у partner_telecom выросло число платежей на подписку?* База канала сжалась в 4,6 раза, а новые подписки продолжают приходить и платят в первый же день. В маленькой базе доля таких «платящих сразу» выше.
- *Почему у остальных каналов влияние активных подписок ровно ноль?* Совпадение: у «без канала» отключили ~600 подписок, а остальные каналы за неделю выросли примерно на столько же. Их −105 тыс. ₽ — колебание частоты списаний, −1,2% выручки.

In [ ]:
# Порядок подстановок — часть ответа. Проверяем все шесть порядков на всей базе.
x0, x1 = to_factors(f0), to_factors(f1)
orders = pd.DataFrame({" → ".join(LABELS[f] for f in order): contrib(x0, x1, list(order))
                       for order in itertools.permutations(ORDER)}).T.rename(columns=LABELS)
orders.loc["среднее по порядкам"] = orders.mean()

(orders / 1000).round(1)       # тыс. ₽

In [ ]:
steps = effects.loc["вся база", list(LABELS.values())] / 1000
ax = waterfall(steps, accent="активные подписки",
               title=f"Выручка упала из-за числа активных подписок ({signed(steps['активные подписки'])} тыс. ₽), "
                     f"частота и цена платежа почти ни при чём")
ax.set_xlabel("Вся база, w0 = 17–21.08 → w1 = 24–28.08. Порядок подстановок — слева направо.")
plt.show()

## 4. Кого задело: витрина отключённых клиентов

**Кого считаю отключённым.** Подписку, которая закончилась 22 или 23 августа **в первый час суток** (00:00–00:59 UTC). Почему так — в разделе 5: в эти два часа закончились десятки тысяч подписок, а в любой другой час тех же суток — единицы, как обычно. Так массовое отключение отделяется от обычного оттока тех же дней.

**Зерно витрины: одна строка — один владелец отключённой подписки.** Число строк объявляю до сборки. Сторону «многие» — подписки, платежи, участников, транзакции — сворачиваю до клиента **до** соединения и объявляю кратность в `validate=`, как на семинаре 4.

**Деньги** — платежи этих подписок за 30 дней до события (23.07–21.08). Списание идёт раз в 30 дней, поэтому такое окно — ровно один цикл списания, то есть месячная выручка, которую эти подписки приносили.

In [ ]:
SWITCHED_OFF = """
    select subscription_id, owner_client_id, tariff_id, is_trial, ended_at
    from prime.subscriptions
    where ended_at >= :e0 and ended_at < :e1
      and extract(hour from ended_at) = 0
"""
EV = {"e0": EVENT[0], "e1": EVENT[1]}
CYCLE = ("2026-07-23", "2026-08-22")       # 30 дней до события = один цикл списания

lost = q(SWITCHED_OFF, **EV)
owners = q(f"""
    select c.client_id, coalesce(c.acquisition_channel, 'без канала') as channel
    from prime.clients c
    where c.client_id in (select owner_client_id from ({SWITCHED_OFF}) x)
    order by c.client_id
""", **EV)

N = len(owners)
print("ЗЕРНО: одна строка — один владелец отключённой подписки.")
print(f"отключено подписок: {len(lost)}; владельцев: {N} — столько строк обязано быть в витрине")

In [ ]:
# Факты, которые сворачиваем до владельца.
pay = q(f"""
    select p.payment_id, p.subscription_id, p.amount
    from prime.payments p
    where p.status = 'success'
      and p.paid_at >= :c0 and p.paid_at < :c1
      and p.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x)
""", c0=CYCLE[0], c1=CYCLE[1], **EV)

members = q(f"""
    select m.subscription_id, m.client_id
    from prime.subscription_members m
    join ({SWITCHED_OFF}) x on x.subscription_id = m.subscription_id
    where m.joined_at < x.ended_at
      and (m.left_at is null or m.left_at >= x.ended_at)    -- был участником в момент отключения
""", **EV)

tx = q(f"""
    select t.client_id,
           count(*) filter (where t.occurred_at < :a1)  as tx_w0,
           count(*) filter (where t.occurred_at >= :b0) as tx_w1
    from prime.transactions t
    where t.status = 'success'
      and t.client_id in (select owner_client_id from ({SWITCHED_OFF}) x)
      and ((t.occurred_at >= :a0 and t.occurred_at < :a1) or (t.occurred_at >= :b0 and t.occurred_at < :b1))
    group by 1
""", a0=W0[0], a1=W0[1], b0=W1[0], b1=W1[1], **EV)

back = q(f"""
    select owner_client_id as client_id, count(*) as new_subs
    from prime.subscriptions
    where started_at >= :e1
      and owner_client_id in (select owner_client_id from ({SWITCHED_OFF}) x)
    group by 1
""", **EV)

print(f"платежей за цикл: {len(pay)}; мест участников: {len(members)}; "
      f"владельцев с покупками у партнёров: {len(tx)}; оформивших новую подписку: {len(back)}")

In [ ]:
def step(name: str, before: pd.DataFrame, after: pd.DataFrame, key: str,
         validate: str = "", expect_lost: int | None = None) -> dict:
    """Одна строка протокола сборки (семинар 4).

    keys_lost — строки слева, которым не нашлось пары справа:
    видно их по пропуску в колонке key, пришедшей из правой таблицы.
    """
    lost_keys = int(after[key].isna().sum())
    if len(after) != len(before):
        verdict = f"ВНИМАНИЕ: строк было {len(before)}, стало {len(after)}"
    elif expect_lost is None:
        verdict = "ok"
    elif lost_keys != expect_lost:
        verdict = f"ВНИМАНИЕ: потеряно {lost_keys}, ждали {expect_lost}"
    else:
        verdict = f"ok, ожидали {expect_lost}"
    return {"step": name, "rows_before": len(before), "rows_after": len(after),
            "validate": validate, "keys_lost": lost_keys, "verdict": verdict}


protocol: list[dict] = []
sub_owner = lost[["subscription_id", "owner_client_id"]]

# 1. Подписки -> владелец: у владельца может быть больше одной отключённой подписки.
subs_by_owner = (lost.groupby("owner_client_id", as_index=False)
                     .agg(n_lost=("subscription_id", "size"), n_trial=("is_trial", "sum")))
mart = owners.merge(subs_by_owner, left_on="client_id", right_on="owner_client_id",
                    how="left", validate="1:1").drop(columns="owner_client_id")
protocol.append(step("подписки -> владелец", owners, mart, "n_lost", "1:1", 0))

# 2. Платежи цикла -> подписка -> владелец.
pay_by_owner = (pay.merge(sub_owner, on="subscription_id", validate="m:1")
                   .groupby("owner_client_id", as_index=False)
                   .agg(n_payments=("payment_id", "size"), paid_cycle=("amount", "sum")))
before = mart
mart = mart.merge(pay_by_owner, left_on="client_id", right_on="owner_client_id",
                  how="left", validate="1:1").drop(columns="owner_client_id")
protocol.append(step("платежи цикла -> владелец", before, mart, "n_payments", "1:1", N - len(pay_by_owner)))

# 3. Участники -> владелец: сколько РАЗНЫХ людей потеряли доступ вместе с ним.
mem_by_owner = (members.merge(sub_owner, on="subscription_id", validate="m:1")
                       .groupby("owner_client_id", as_index=False)
                       .agg(n_members=("client_id", "nunique")))
before = mart
mart = mart.merge(mem_by_owner, left_on="client_id", right_on="owner_client_id",
                  how="left", validate="1:1").drop(columns="owner_client_id")
protocol.append(step("участники -> владелец", before, mart, "n_members", "1:1", N - len(mem_by_owner)))

# 4. Транзакции у партнёров: в SQL уже свёрнуты до клиента.
before = mart
mart = mart.merge(tx, on="client_id", how="left", validate="1:1")
protocol.append(step("транзакции -> владелец", before, mart, "tx_w0", "1:1", N - len(tx)))

# 5. Новые подписки после события.
before = mart
mart = mart.merge(back, on="client_id", how="left", validate="1:1")
protocol.append(step("новые подписки -> владелец", before, mart, "new_subs", "1:1", N - len(back)))

counts = ["n_payments", "paid_cycle", "n_members", "tx_w0", "tx_w1", "new_subs"]
mart[counts] = mart[counts].fillna(0)
mart.head()

In [ ]:
# Независимые контроли: считаются в SQL другим путём, без витрины.
money_control = q("""
    select sum(p.amount)
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= :c0 and p.paid_at < :c1
      and s.ended_at >= :e0 and s.ended_at < :e1 and extract(hour from s.ended_at) = 0
""", c0=CYCLE[0], c1=CYCLE[1], **EV).iloc[0, 0]

people_control = q(f"""
    select count(*) from (
        select owner_client_id as client_id from ({SWITCHED_OFF}) x
        union
        select m.client_id
        from prime.subscription_members m
        join ({SWITCHED_OFF}) x on x.subscription_id = m.subscription_id
        where m.joined_at < x.ended_at and (m.left_at is null or m.left_at >= x.ended_at)
    ) u
""", **EV).iloc[0, 0]

# Ломаем намеренно: деньги через subscription_members, как делать нельзя.
naive = q(f"""
    select sum(p.amount)
    from prime.payments p
    join prime.subscription_members m on m.subscription_id = p.subscription_id
    where p.status = 'success' and p.paid_at >= :c0 and p.paid_at < :c1
      and p.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x)
""", c0=CYCLE[0], c1=CYCLE[1], **EV).iloc[0, 0]

grain_ok = len(mart) == mart["client_id"].nunique() == N
money_diff = round(mart["paid_cycle"].sum() - money_control, 2)
people = len(set(mart["client_id"]) | set(members["client_id"]))
seats = int(mart["n_members"].sum())

print(pd.DataFrame(protocol).to_string(index=False))
print()
print(f"ЗЕРНО   строк {len(mart)}, клиентов {mart['client_id'].nunique()}, объявляли {N}"
      f"   -> {'ok' if grain_ok else 'ВНИМАНИЕ'}")
print(f"ДЕНЬГИ  разница с независимым контролем: {money_diff:.2f} ₽"
      f"   -> {'ok' if money_diff == 0 else 'ВНИМАНИЕ'}")
print(f"ЛЮДИ    по витрине {people}, независимо {people_control}"
      f"   -> {'ok' if people == people_control else 'ВНИМАНИЕ'}")
print()
print(f"Через subscription_members вышло бы {rub(naive)} ₽ вместо {rub(money_control)} ₽ — "
      f"в {naive / money_control:.2f} раза больше: двойной счёт из-за шеринга.")
assert grain_ok and money_diff == 0 and people == people_control, "протокол не сошёлся"

In [ ]:
company_cycle = q("""
    select sum(amount) from prime.payments
    where status = 'success' and paid_at >= :c0 and paid_at < :c1
""", c0=CYCLE[0], c1=CYCLE[1]).iloc[0, 0]

impact = pd.Series({
    "отключено подписок": len(lost),
    "  из них пробных": lost["is_trial"].sum(),
    "владельцев": N,
    "  из них канал partner_telecom, %": (mart["channel"] == "partner_telecom").mean() * 100,
    "участников, потерявших доступ (людей)": members["client_id"].nunique(),
    "людей всего: владельцы и участники": people,
    "выручка этих подписок за 30 дней до события, ₽": mart["paid_cycle"].sum(),
    "доля в выручке ПРАЙМ за те же 30 дней, %": mart["paid_cycle"].sum() / company_cycle * 100,
    "владельцев с покупками у партнёров в w0, %": (mart["tx_w0"] > 0).mean() * 100,
    "владельцев с покупками у партнёров в w1, %": (mart["tx_w1"] > 0).mean() * 100,
    "оформили новую подписку после 23.08": (mart["new_subs"] > 0).sum(),
})
pd.DataFrame({"значение": [f"{v:.1f}" if "%" in k else rub(v) for k, v in impact.items()]},
             index=impact.index)

In [ ]:
subs_daily = q(f"""
    with own as (select distinct owner_client_id as client_id from ({SWITCHED_OFF}) x)
    select d::date as day, count(s.subscription_id) as subs
    from generate_series(cast('2026-08-01' as timestamptz), cast(:last as timestamptz) - interval '1 day',
                         interval '1 day') as d
    left join prime.subscriptions s
      on s.owner_client_id in (select client_id from own)
     and s.started_at < d + interval '1 day'
     and (s.ended_at is null or s.ended_at >= d + interval '1 day')   -- активна на конец дня
    group by 1
    order by 1
""", last=LAST_DAY, **EV)

tx_daily = q(f"""
    select t.occurred_at::date as day, count(*) as tx
    from prime.transactions t
    where t.status = 'success'
      and t.occurred_at >= '2026-08-01' and t.occurred_at < :last
      and t.client_id in (select owner_client_id from ({SWITCHED_OFF}) x)
    group by 1
    order by 1
""", last=LAST_DAY, **EV)

eco = subs_daily.merge(tx_daily, on="day", how="left", validate="1:1")
eco["day"] = pd.to_datetime(eco["day"])
eco = eco.set_index("day")

# Две разные величины на одной оси нельзя — приводим обе к индексу: среднее за w0 = 100.
base = eco[(eco.index >= W0[0]) & (eco.index < W0[1])].mean()
idx = eco / base * 100

fig, ax = plt.subplots(figsize=(10, 4.5), layout="constrained")
ax.plot(idx.index, idx["tx"], color=GREY, linewidth=2, label="покупки у партнёров в день")
ax.plot(idx.index, idx["subs"], color=ACCENT, linewidth=2, label="активные подписки на конец дня")
ax.axvspan(pd.Timestamp(EVENT[0]), pd.Timestamp(EVENT[1]), color=ACCENT, alpha=0.12, linewidth=0)
ax.set_ylim(0, 125)
ax.set_ylabel("индекс, среднее за 17–21.08 = 100")
ax.set_xlabel("Владельцы отключённых подписок, август 2026, UTC")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d.%m"))
w1_mask = (idx.index >= W1[0]) & (idx.index < W1[1])
ax.set_title(f"Подписок у этих клиентов осталось {idx.loc[w1_mask, 'subs'].mean():.0f}% от прежнего, "
             f"а покупок у партнёров — {idx.loc[w1_mask, 'tx'].mean():.0f}%: из экосистемы они не ушли", loc="left")
ax.legend(frameon=False, loc="center left")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color=GRID)
plt.show()

## 5. Почему — и чего мы не знаем

Разложение отвечает «за счёт чего», а не «почему» (семинар 6). Ниже факты, которые сужают круг причин.

In [ ]:
# Во сколько заканчивались подписки 22–23 августа.
hours = q("""
    select ended_at::date as day, extract(hour from ended_at)::int as hour, count(*) as n
    from prime.subscriptions
    where ended_at >= :e0 and ended_at < :e1
    group by 1, 2
""", **EV).pivot(index="hour", columns="day", values="n").fillna(0)

pd.DataFrame({"00:00–00:59": hours.loc[0], "в среднем за любой другой час": hours.drop(index=0).mean()}).round(1)

In [ ]:
# Бывало ли такое раньше: самые «тяжёлые» часы по числу отмен за всю историю базы.
q("""
    select date_trunc('hour', ended_at) as hour, count(*) as ended
    from prime.subscriptions
    where ended_at is not null
    group by 1
    order by 2 desc
    limit 5
""")

In [ ]:
# Какую долю активной базы каждого канала отключили.
reach = q(f"""
    select coalesce(c.acquisition_channel, 'без канала') as channel,
           count(*) filter (where s.started_at < :e0
                              and (s.ended_at is null or s.ended_at >= :e0))                        as active_before,
           count(*) filter (where s.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x)) as switched_off
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    group by 1
    order by 3 desc
""", **EV).set_index("channel")
reach["switched_off_%"] = (reach["switched_off"] / reach["active_before"] * 100).round(1)
reach

In [ ]:
# «Без канала» отключили 15%. Похоже ли это на клиентов оператора, у которых потерялась метка канала?
# Если пустой канал раскидан по клиентам случайно, среди них partner_telecom столько же, сколько в базе с известным каналом.
known = reach.drop(index="без канала")
tel_share = known.loc["partner_telecom", "active_before"] / known["active_before"].sum()
expected = tel_share * reach.loc["partner_telecom", "switched_off_%"]

print(f"доля partner_telecom среди активных подписок с известным каналом: {tel_share:.1%}")
print(f"если «без канала» — случайная выборка, отключить должны были ≈ {expected:.1f}%, "
      f"на деле {reach.loc['без канала', 'switched_off_%']:.1f}%")

In [ ]:
# Чем отключённые подписки partner_telecom отличаются от оставшихся? Доля отключённых, %, по признакам.
tel_subs = q(f"""
    select s.tariff_id, s.is_trial, extract(year from s.started_at)::int as start_year,
           c.kyc_level, s.partner_id,
           s.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x) as switched_off
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    where c.acquisition_channel = 'partner_telecom'
      and s.started_at < :e0 and (s.ended_at is null or s.ended_at >= :e0)
""", **EV)

pd.concat({col: tel_subs.groupby(col)["switched_off"].mean().mul(100).round(1)
           for col in ["tariff_id", "is_trial", "start_year", "kyc_level", "partner_id"]}).to_frame("отключено, %")

In [ ]:
# Причины отмены у отключённых и у обычного оттока июля, в процентах.
reasons = q(f"""
    select case when s.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x)
                then 'отключение 22–23.08' else 'обычный отток, июль 2026' end as grp,
           coalesce(s.cancel_reason, 'не указана') as reason,
           count(*) as n
    from prime.subscriptions s
    where (s.ended_at >= '2026-07-01' and s.ended_at < '2026-08-01')
       or s.subscription_id in (select subscription_id from ({SWITCHED_OFF}) x)
    group by 1, 2
""", **EV).pivot(index="reason", columns="grp", values="n")

(reasons / reasons.sum() * 100).round(1)

In [ ]:
# Обращения в поддержку и маркетинг по окнам.
WIN = {"a0": W0[0], "a1": W0[1], "b0": W1[0], "b1": W1[1], "e0": EVENT[0]}

tickets = q("""
    select category, case when created_at < :e0 then 'w0' else 'w1' end as w, count(*) as n
    from prime.support_tickets
    where (created_at >= :a0 and created_at < :a1) or (created_at >= :b0 and created_at < :b1)
    group by 1, 2
""", **WIN).pivot(index="category", columns="w", values="n")
tickets["delta"] = tickets["w1"] - tickets["w0"]
display(tickets)

marketing = q("""
    with spend as (
        select channel, case when spend_date < :e0 then 'w0' else 'w1' end as w, sum(amount) as spend
        from prime.marketing_spend
        where (spend_date >= :a0 and spend_date < :a1) or (spend_date >= :b0 and spend_date < :b1)
        group by 1, 2
    ),
    new as (
        select coalesce(c.acquisition_channel, 'без канала') as channel,
               case when s.started_at < :e0 then 'w0' else 'w1' end as w,
               count(*) as new_subs
        from prime.subscriptions s
        join prime.clients c on c.client_id = s.owner_client_id
        where (s.started_at >= :a0 and s.started_at < :a1) or (s.started_at >= :b0 and s.started_at < :b1)
        group by 1, 2
    )
    select n.channel, n.w, sp.spend, n.new_subs
    from new n
    left join spend sp on sp.channel = n.channel and sp.w = n.w
""", **WIN)
marketing["spend_per_new_sub"] = marketing["spend"] / marketing["new_subs"]
marketing.pivot(index="channel", columns="w", values=["spend", "new_subs", "spend_per_new_sub"]).round(0)

In [ ]:
# Работает ли канал после события: регистрации, новые подписки, расходы и клики — в среднем за день.
channel_daily = q("""
    select d::date as day,
           (select count(*) from prime.clients c
             where c.acquisition_channel = 'partner_telecom'
               and c.registered_at >= d and c.registered_at < d + interval '1 day')  as registered,
           (select count(*) from prime.subscriptions s
              join prime.clients c on c.client_id = s.owner_client_id
             where c.acquisition_channel = 'partner_telecom'
               and s.started_at >= d and s.started_at < d + interval '1 day')        as new_subs,
           (select sum(m.amount) from prime.marketing_spend m
             where m.channel = 'partner_telecom' and m.spend_date = d::date)          as spend,
           (select sum(m.clicks) from prime.marketing_spend m
             where m.channel = 'partner_telecom' and m.spend_date = d::date)          as clicks
    from generate_series(cast(:a as timestamptz), cast(:b as timestamptz) - interval '1 day',
                         interval '1 day') as d
""", a=W0[0], b=W1[1])
channel_daily["day"] = pd.to_datetime(channel_daily["day"])

period = pd.cut(channel_daily["day"], bins=pd.to_datetime([W0[0], EVENT[0], EVENT[1], W1[1]]), right=False,
                labels=["w0: 17–21.08", "событие: 22–23.08", "w1: 24–28.08"])
channel_daily.groupby(period, observed=True)[["registered", "new_subs", "spend", "clicks"]].mean().round(0)

In [ ]:
# Новые подписки канала, оформленные после события: оплачиваются ли они, не отключают ли их.
q("""
    select count(distinct s.subscription_id)                                         as new_subs,
           count(distinct s.subscription_id) filter (where s.is_trial)               as trials,
           count(distinct p.subscription_id)                                         as paid,
           count(distinct s.subscription_id) filter (where s.ended_at is not null)   as ended
    from prime.subscriptions s
    join prime.clients c on c.client_id = s.owner_client_id
    left join prime.payments p on p.subscription_id = s.subscription_id and p.status = 'success'
    where c.acquisition_channel = 'partner_telecom' and s.started_at >= :e1
""", **EV)

In [ ]:
# Не было ли перед отключением волны неуспешных списаний у канала — то есть автоотмены за долги?
fails = q("""
    select p.paid_at::date as day, p.status, count(*) as n
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    join prime.clients c on c.client_id = s.owner_client_id
    where c.acquisition_channel = 'partner_telecom'
      and p.paid_at >= '2026-08-08' and p.paid_at < :e0
    group by 1, 2
""", **EV).pivot(index="day", columns="status", values="n")
fails["failed_%"] = (fails["failed"] / fails.sum(axis=1) * 100).round(1)
fails

In [ ]:
# Вдруг доступ на деле не отключился? Сессии сервисов за 5 дней до окончания подписки и за дни 2–6 после —
# у отключённых и у обычного оттока. Берём только тех, у кого не осталось другой своей активной подписки.
q("""
    with ended as (
        select s.owner_client_id as client_id, s.ended_at,
               case when s.ended_at >= :e0 and s.ended_at < :e1 and extract(hour from s.ended_at) = 0
                    then 'отключение 22–23.08' else 'обычный отток 01.07–15.08' end as grp
        from prime.subscriptions s
        where (s.ended_at >= '2026-07-01' and s.ended_at < '2026-08-16')
           or (s.ended_at >= :e0 and s.ended_at < :e1 and extract(hour from s.ended_at) = 0)
    ),
    alone as (
        select e.* from ended e
        where not exists (
            select 1 from prime.subscriptions s2
            where s2.owner_client_id = e.client_id
              and s2.started_at <= e.ended_at + interval '6 days'
              and (s2.ended_at is null or s2.ended_at > e.ended_at)
        )
    )
    select a.grp,
           count(distinct a.client_id) as clients,
           count(u.usage_id) filter (where u.started_at < a.ended_at)                     as sessions_before,
           count(u.usage_id) filter (where u.started_at >= a.ended_at + interval '1 day') as sessions_after
    from alone a
    left join prime.service_usage u
      on u.client_id = a.client_id
     and ((u.started_at >= a.ended_at - interval '5 days' and u.started_at < a.ended_at)
       or (u.started_at >= a.ended_at + interval '1 day' and u.started_at < a.ended_at + interval '6 days'))
    group by 1
""", **EV).assign(after_to_before=lambda d: (d["sessions_after"] / d["sessions_before"]).round(2))

**Что следует из раздела 5.**

- **Время.** Подписки закончились двумя пачками ровно в полночь: 13,7 тыс. 22.08 и 14,0 тыс. 23.08. В любой другой час тех же суток — около десяти, а за всю историю базы нет ни одного другого часа больше чем с 22 отменами. Клиенты так не отменяют: это разовая пакетная операция.
- **Кого.** 79% активной базы partner_telecom. Ещё 15% «без канала» — почти ровно столько, сколько ждёшь, если среди них ~18% клиентов оператора с потерянной меткой канала (18% × 79% ≈ 14%). В остальных каналах — единицы.
- **По какому признаку.** Ни по какому из тех, что есть в базе: доля отключённых одинакова при любом тарифе, пробном периоде, годе подключения, уровне KYC и партнёре.
- **Канал при этом работает.** Регистрации, новые подписки, расходы и клики после события на прежнем уровне, с обычными колебаниями. Новые подписки канала оплачиваются, ни одна не отменена. Волны неуспешных списаний перед событием не было — это не автоотмена за долги.
- **Причина отмены.** `cancel_reason` у отключённых распределена так же, как у обычного оттока, — поле заполнено формально и про событие ничего не говорит.
- **Реакция клиентов.** Обращений в поддержку не прибавилось, повторных подписок нет. Сервисами после отключения пользуются примерно так же мало, как обычные отписавшиеся, — версию «доступ на деле остался» данные не подтверждают.

**Версия.** Партнёр программу не закрыл: он продолжает приводить клиентов, и их подписки работают. Клиенты не уходили сами. Остаётся разовая пакетная операция над уже существующей базой канала — например, ошибочная выгрузка или синхронизация списка отключений с оператором. Это гипотеза: подтвердить её могут только логи интеграции или ответ партнёра, в базе их нет.

## 6. Вывод

**Что случилось.** В неделю 24–28 августа 2026 года выручка ПРАЙМ — 9,03 млн ₽ против 10,43 млн ₽ неделей раньше (17–21.08, те же дни недели): **−1,41 млн ₽, или −13,5%.** В первый час суток 22 и 23 августа закончились 27 689 подписок — при обычном фоне около десяти в час.

**Где и за счёт чего.** **92% падения — канал partner_telecom**: его выручка упала на 72% (−1,30 млн ₽), остальные каналы изменились на −1,2%. Внутри — **число активных подписок**: −26,9 тыс. (−13,8%) дало −1,44 млн ₽. Частота списаний (+9 тыс. ₽) и цена платежа (+19 тыс. ₽) почти ни при чём, и это не зависит от порядка подстановок. Если подписки не вернутся, ПРАЙМ теряет **8,5 млн ₽ в месяц — 13,7% выручки**. Доступ потеряли **63,5 тыс. человек**: 27,7 тыс. владельцев и 37,9 тыс. участников их подписок.

**Чего мы не знаем.** Причины в данных нет. Самая правдоподобная версия — разовая пакетная операция над старой базой канала, например ошибочная синхронизация списка отключений с оператором: две пачки ровно в полночь, такого не было ни разу за всю историю, и отбор не связан ни с одним клиентским признаком. Закрытие программы партнёром данные не подтверждают: канал по-прежнему приводит ~60 подписок в день, и они оплачиваются. Клиенты, похоже, ещё не заметили: обращений не прибавилось, повторных подписок нет. Данные кончаются 29.08, поэтому неизвестно, вернутся ли клиенты сами и не отключат ли оставшиеся ~7 тыс. подписок канала.

**Что делать.**

1. **Сразу — партнёрский менеджер и техподдержка.** Запросить у оператора, что ушло в обмен данными в 00:00 22 и 23 августа. Канал работает, поэтому вероятнее сбой, чем расторжение: восстановить 27,7 тыс. подписок по прежней цене и без повторного списания, пока клиенты не заметили.
2. **Если отключение окажется намеренным — CRM, в течение недели.** Кампания возврата по витрине `mart` с предложением перейти на прямую оплату. Начать с 26,1 тыс. владельцев, плативших в последний цикл, а среди них в первую очередь — с тех, у кого есть участники (всего таких владельцев 18,2 тыс.): вместе с владельцем возвращается и семья.
3. **Маркетинг — бюджет не останавливать, но и не наращивать.** Новые подписки канала оформляются и оплачиваются как раньше (~60 в день, ~1 000 ₽ расходов на подписку), так что останавливать расходы (~1,9 млн ₽ в месяц) незачем. Наращивать — только после ответа оператора: пока причина неизвестна, нельзя исключить, что новых клиентов отключат так же.
4. **Аналитика — постоянно.** Ежедневная проверка числа отмен по часам и каналам с порогом: обычный фон — около 10 отмен в час, здесь было 13 700.